In [58]:
def calculate_matrix_mean(matrix: list[list[float]], mode: str) -> list[float]:
    means = []
    aux = 0
    if mode == "column":
        for i in range(len(matrix[0])):
            for j in range(len(matrix)):
                print('i: ', i)
                print('j: ', j)
                print('matrix: ', matrix[j][i])
                aux += matrix[j][i]
            means.append(aux/len(matrix))
            aux = 0
    elif mode == "row":
        for i in range(len(matrix)):
            for j in range(len(matrix[i])):
                aux += matrix[i][j]
            means.append(aux/len(matrix[i]))
            aux = 0
    return means

In [60]:
print(calculate_matrix_mean([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 'column'))
print(calculate_matrix_mean([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 'row'))
print(calculate_matrix_mean([[1, 2, 3, 4], [5, 6, 7, 8]], 'row'))
print(calculate_matrix_mean([[1, 2, 3, 4], [5, 6, 7, 8]], 'column'))



i:  0
j:  0
matrix:  1
i:  0
j:  1
matrix:  4
i:  0
j:  2
matrix:  7
i:  1
j:  0
matrix:  2
i:  1
j:  1
matrix:  5
i:  1
j:  2
matrix:  8
i:  2
j:  0
matrix:  3
i:  2
j:  1
matrix:  6
i:  2
j:  2
matrix:  9
[4.0, 5.0, 6.0]
[2.0, 5.0, 8.0]
[2.5, 6.5]
i:  0
j:  0
matrix:  1
i:  0
j:  1
matrix:  5
i:  1
j:  0
matrix:  2
i:  1
j:  1
matrix:  6
i:  2
j:  0
matrix:  3
i:  2
j:  1
matrix:  7
i:  3
j:  0
matrix:  4
i:  3
j:  1
matrix:  8
[3.0, 4.0, 5.0, 6.0]


In [67]:
PATH_PEDIDOS=r"C:\Users\felipe.quentino_quer\Documents\qd-models\features\raw_data\pedido_202610011142.csv"
PATH_MONGO=r"C:\Users\felipe.quentino_quer\Documents\qd-models\features\raw_data\raio-x.csv"
import pandas as pd 
df_dw = pd.read_csv(PATH_PEDIDOS)
df_mongo = pd.read_csv(PATH_MONGO)
df_dw.head()

,pedidoId,usuarioId,placeId,municipioId,precoTotal,precoProdutos,status,createdAt
0,6a4482cb2c79e82a9e0782a1,6a0a896b6291f5757a2ecd15,6287ce9bb1b83600e07114ad,5e1c62a63eb4ec0032661fbd,39.50,35.00,ENTREGUE,2026-07-01 00:00:00
1,6a4482ce875763f23d1577b5,60b6f0e1bc7d6a0045ebaaa3,61fe63f1b7992500b5b62fd3,5ca79a58c36ec6001bf5974c,39.48,33.98,ENTREGUE,2026-07-01 00:00:00
2,6a4482b210c27e9cde916b6a,5cd49cc5167bc500b5e99b91,5bf843c4740c9a003322b942,5b5881d05c6347001ff0d192,33.99,33.99,ENTREGUE,2026-07-01 00:00:00
3,6a4482d82a1280670f23e143,5c5e2cbba0a4ea002140c737,685d4083b8aaea166056ac6a,5c0548f594879f0014ff5825,88.50,80.00,ENTREGUE,2026-07-01 00:00:00
4,6a4482b781231a95fe442726,6871a050df634a18c86af01e,606a3d2a3705b50083bea20d,5cbf908c07875a00885ccd1d,31.50,25.00,ENTREGUE,2026-07-01 00:00:00


In [66]:
# prova real

gmv = df_dw.precoTotal.sum()
qtd_pedidos = df_dw.pedidoId.count()
users_distinct = df_dw.usuarioId.nunique()
print(qtd_pedidos)
print(gmv)
print(users_distinct)

1125522
50372917.220000006
286861


In [68]:
print(df_dw.shape)
print(df_mongo.shape)

(1125522, 8)
(286861, 19)


In [72]:
df_mongo.qtdPedidos.sum()

np.int64(1125522)

In [76]:
df_dw.precoTotal.sum()

np.float64(50372917.220000006)

In [71]:
df_mongo.gmv.sum()

np.int64(50377567)

In [70]:
print(df_mongo.columns)
print(df_dw.columns)

Index(['_id', 'usuarioId', 'periodoAnaliseStart', 'periodoAnaliseEnd',
       'createdAt', 'qtdPedidos', 'ticketMedio', 'gmv', 'pedidosNaoEntregues',
       'pedidosRejeitados', 'municipioNome', 'pedidosMes[0].mes',
       'pedidosMes[0].ano', 'pedidosMes[0].qtdPedidos',
       'pedidosMes[0].qtdPedido_Frete_Gratis',
       'pedidosMes[0].qtdPedido_Com_Coupon',
       'pedidosMes[0].qtdPedido_Geraram_CashBack',
       'pedidosMes[0].qtsCategoriaCompram', 'municipioId'],
      dtype='str')
Index(['pedidoId', 'usuarioId', 'placeId', 'municipioId', 'precoTotal',
       'precoProdutos', 'status', 'createdAt'],
      dtype='str')


In [73]:
50372917.220000006 - 50377567

-4649.7799999937415

In [74]:
import numpy as np

# GMV e pedidos por usuário no DW
dw_user = (
    df_dw.groupby('usuarioId')
         .agg(gmv_dw=('precoTotal', 'sum'), pedidos_dw=('pedidoId', 'count'))
         .reset_index()
)

mongo_user = df_mongo[['usuarioId', 'gmv', 'qtdPedidos']].rename(
    columns={'gmv': 'gmv_mongo', 'qtdPedidos': 'pedidos_mongo'}
)

# garante o mesmo tipo de chave (ObjectId vs string costuma quebrar o merge)
dw_user['usuarioId'] = dw_user['usuarioId'].astype(str)
mongo_user['usuarioId'] = mongo_user['usuarioId'].astype(str)

comp = dw_user.merge(mongo_user, on='usuarioId', how='outer', indicator=True)

# 1) usuários que só existem de um lado
print(comp['_merge'].value_counts())

comp['diff'] = comp['gmv_mongo'] - comp['gmv_dw']
comp['diff_pedidos'] = comp['pedidos_mongo'] - comp['pedidos_dw']

# 2) quanto da diferença o arredondamento explica
comp['diff_arred'] = comp['gmv_mongo'] - comp['gmv_dw'].round()
print('Diferença total:', comp['diff'].sum())
print('Diferença após arredondar o DW:', comp['diff_arred'].sum())

# 3) casos que NÃO são arredondamento (diferença >= 1 real)
fora = comp[comp['diff'].abs() >= 1].sort_values('diff', key=abs, ascending=False)
print('Usuários com diferença real:', len(fora))
print(fora.head(20))

# 4) usuários com quantidade de pedidos diferente
print('Pedidos divergentes:', (comp['diff_pedidos'] != 0).sum())

_merge
both          286861
left_only          0
right_only         0
Name: count, dtype: int64
Diferença total: 4649.780000000041
Diferença após arredondar o DW: 29645.0
Usuários com diferença real: 0
Empty DataFrame
Columns: [usuarioId, gmv_dw, pedidos_dw, gmv_mongo, pedidos_mongo, _merge, diff, diff_pedidos, diff_arred]
Index: []
Pedidos divergentes: 0


In [75]:
dw = comp['gmv_dw'].round(2)  # remove o lixo de ponto flutuante primeiro
mongo = comp['gmv_mongo']

regras = {
    'half-up (comum)': np.floor(dw + 0.5),
    'half-even (pandas)': dw.round(),
    'ceil (pra cima)': np.ceil(dw),
    'floor/truncar': np.floor(dw),
}

for nome, valor in regras.items():
    print(f'{nome:20s} bate em {(valor == mongo).mean():.4%} | diff total: {(mongo - valor).sum()}')

# distribuição das diferenças por usuário
print(comp['diff'].round(2).describe())

half-up (comum)      bate em 100.0000% | diff total: 0.0
half-even (pandas)   bate em 89.6657% | diff total: 29645.0
ceil (pra cima)      bate em 69.6940% | diff total: -86936.0
floor/truncar        bate em 47.5171% | diff total: 150553.0
count    286861.000000
mean          0.016209
std           0.330008
min          -0.490000
25%          -0.260000
50%           0.010000
75%           0.250000
max           0.500000
Name: diff, dtype: float64


C  = Pi +  Ki  + mod 26
Pi = Ci -  Ki  + mod 26

In [77]:
palavra = "artyom"
alphabet = "abcdefghijklmnopqrstuvwxyz"
for i in range in palavra:
    posicao = alphabet.find("p")
    print(p)

a
r
t
y
o
m


In [147]:
from math import sqrt

def crypto_vigenere(word, key):
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    crypto = ''
    aux = 0
    for i in range(len(word)):
        p = word[i]
        if aux > len(key) - 1:
            aux = 0 
        k = key[aux]
        crypto += alphabet[((alphabet.find(p) + alphabet.find(k)) % 26)]
        aux += 1
    return crypto

def decrypto_vigenere(word, crypto):
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    key = ''
    for i in range(len(word)):
        p = word[i] 
        c = crypto[i]
        key += alphabet[((alphabet.find(c) - alphabet.find(p) + 26) % 26)]

    return key

def swap(word):
    swapped_word = list(word)
    start = 1
    end = len(word) - 2
    while start != end:
        aux = swapped_word[end]
        swapped_word[end] = swapped_word[start]
        swapped_word[start] = aux
        start += 1
        end -= 1
    return "".join(swapped_word)

def is_prime(n):
    if n % 2 == 0:
        return False
    squared_root = round(sqrt(n))
    for i in range(squared_root, 2, -1):
        if n % i == 0:
            return False
    # none of the up cases passed, so, the number is prime
    return True
    
def clip_consonants_vogals(word):
    vogais = 'aeiou'
    resultado_vogais = ''
    resultado_cw = ''
    sz = len(word)
    for letter in word:
        if letter in vogais:
            resultado_vogais += letter
        else:
            resultado_cw += letter
            
    if is_prime(sz):
        return resultado_cw
        
    return resultado_vogais
        
    
    
        

In [148]:
def misterio(word, key):
    vigenere = crypto_vigenere(word, key)
    swapped = swap(word)
    clipped = clip_consonants_vogals(word)
    return vigenere+swapped+clipped

In [149]:
misterio("bussola", "artyom")

bllqcxa
blossua
bssl


'bllqcxablossuabssl'

In [140]:
clip_consonants_vogals("bussol")

'uo'

In [134]:
is_prime(101)

True

In [88]:
crypto_vigenere("bussola", "artyom")

'bllqcxa'

In [95]:
decrypto_vigenere("bussola", "bllqcxa")

'artyoma'

In [117]:
swap("bussola")

['b', 'l', 'o', 's', 's', 'u', 'a']